# 01 — Exploration des données (EDA)

**Projet** : Diagnostic du mode de défaillance machine avec LightGBM
**Cas métier** : Prédire, au moment de l'alarme et à partir des seuls capteurs disponibles, le mode de défaillance parmi six (fausse alarme, usure outil, dissipation thermique, défaut de puissance, surcharge mécanique, défaillance aléatoire), afin d'envoyer directement la bonne équipe — ou de demander un avis expert quand le modèle doute.
**Jeu de données** : `machine_failure_diagnosis` — Alarmes machine et mode de défaillance diagnostiqué

> Une ligne par alarme levée par une fraiseuse CNC : contexte de production, code émis par l'automate, relevés capteurs à l'instant de l'alarme, et mode de défaillance constaté par le technicien à la clôture de l'intervention. Le mode est tiré d'un modèle physique (usure, écart thermique à basse vitesse, puissance hors plage, effort au-delà d'un seuil dépendant de la gamme) avec un bruit irréductible : le signal est réel mais imparfait, et une classe (`random_failure`) n'a volontairement aucun signal observable.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Comprendre l'objectif softmax d'un booster : un arbre par classe et par itération.
- Armer l'early stopping natif sur une métrique multi-classes (`multi_logloss`).
- Garder des arbres petits pour préserver la calibration, dont dépend la décision à coût minimal.

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (6000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 6000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.55)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `machine_failure_diagnosis` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Valeurs manquantes volontaires sur `vibration_mm_s` (~4 %) pour exercer l'imputation.; Les classes sont tirées d'un softmax de logits physiques calculés sur les seules observables ; les ordonnées sont calibrées pour tenir les parts visées..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. La cible : six modes de défaillance

Une cible multi-classes ne se résume pas à un taux : il faut savoir **combien** pèse chaque mode
(le déséquilibre fixe la métrique), **ce qui les sépare** (quelles combinaisons de capteurs), et
**pourquoi la règle actuelle se trompe** (c'est la valeur que le modèle doit apporter).

In [ ]:
target = CONFIG.data.target
order = [
    label for label in CONFIG.model_dump()["diagnosis"]["class_order"] if label in set(raw[target])
]
counts = raw[target].value_counts().reindex(order)
shares = counts / counts.sum()

fig, axis = plt.subplots(figsize=(8.6, 3.4))
axis.bar(order, counts, color="#0a9396", edgecolor="white")
for position, (count, share) in enumerate(zip(counts, shares, strict=True)):
    axis.text(position, count, f"{share:.1%}", ha="center", va="bottom", fontsize=8)
axis.set_title(f"Répartition de `{target}` ({len(raw)} alarmes)")
axis.set_ylabel("alarmes")
axis.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()

majority_f1 = shares.max() / (1 + shares.max()) * 2 / len(order)
print(f"rapport majoritaire / minoritaire : {counts.max() / counts.min():.1f}")
print(f"accuracy de la classe majoritaire : {shares.max():.1%}")
print(f"macro-F1 de la classe majoritaire : {majority_f1:.3f}")

**Ce qu'il faut retenir**

- Six classes de 30 % à 6 % : répondre toujours « fausse alarme » donne 30 % d'accuracy mais un macro-F1 proche de 0,08. C'est pour cela que le **macro-F1** pilote ce projet : chaque mode compte autant, quelle que soit sa fréquence.
- Le mode le plus rare (`random_failure`) ne pèse que quelques centaines d'alarmes : ses métriques seront les plus bruitées, et le split stratifié est obligatoire pour qu'il existe dans chaque partie.
- La métrique n'est pas un détail technique : un modèle réglé sur l'accuracy apprendrait à ignorer les modes rares, qui sont justement ceux qui coûtent cher à rater.

In [ ]:
# Ce que dit le code automate, confronté au mode réellement constaté (part de chaque code).
alarm_column = CONFIG.model_dump()["diagnosis"]["alarm_code_column"]
routing = (
    pd.crosstab(raw[alarm_column], raw[target], normalize="index").reindex(columns=order).round(3)
)
rule = CONFIG.model_dump()["diagnosis"]["alarm_code_rule"]
routing.insert(0, "mode visé par la règle", [rule.get(code, "-") for code in routing.index])
routing["exact"] = [
    routing.loc[code, rule[code]] if rule.get(code) in routing.columns else float("nan")
    for code in routing.index
]
routing

**Ce qu'il faut retenir**

- La colonne `exact` est la part des alarmes d'un code pour lesquelles la règle envoie la bonne équipe : c'est la **précision de la règle actuelle**, code par code.
- Le code générique `E901` mélange fausses alarmes et vraies pannes, dont toutes les défaillances aléatoires : la règle les acquitte toutes, ce qui est l'erreur la plus chère du métier.
- Les règles de l'automate ignorent la gamme de produit : le seuil de surcharge réel dépend pourtant de la gamme (L/M/H), d'où des surcharges mal routées sur les pièces haute précision.

In [ ]:
# Les modes ne se séparent pas sur un capteur, mais sur des combinaisons physiques.
frame = raw.dropna(subset=["torque_nm"]).sample(n=min(2500, len(raw)), random_state=CONFIG.seed)
gap = frame["process_temperature_k"] - frame["air_temperature_k"]
power = frame["torque_nm"] * frame["rotational_speed_rpm"] * 2 * np.pi / 60
strain = frame["tool_wear_min"] * frame["torque_nm"]
palette = dict(zip(order, plt.cm.tab10.colors, strict=False))
colours = frame[target].map(palette)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))
axes[0].scatter(frame["rotational_speed_rpm"], gap, c=colours, s=6, alpha=0.6)
axes[0].axhline(8.6, color="#9b2226", linestyle="--", linewidth=1)
axes[0].axvline(1380, color="#9b2226", linestyle="--", linewidth=1)
axes[0].set_xlabel("vitesse (tr/min)")
axes[0].set_ylabel("écart process - air (K)")
axes[0].set_title("Dissipation thermique : écart < 8,6 K à basse vitesse")
axes[1].scatter(frame["rotational_speed_rpm"], power, c=colours, s=6, alpha=0.6)
axes[1].axhline(3500, color="#9b2226", linestyle="--", linewidth=1)
axes[1].axhline(9000, color="#9b2226", linestyle="--", linewidth=1)
axes[1].set_xlabel("vitesse (tr/min)")
axes[1].set_ylabel("puissance ≈ couple x vitesse (W)")
axes[1].set_title("Défaut de puissance : hors [3 500, 9 000] W")
axes[2].scatter(frame["tool_wear_min"], strain, c=colours, s=6, alpha=0.6)
axes[2].axvline(200, color="#9b2226", linestyle="--", linewidth=1)
for quality, threshold in {"L": 11000, "M": 12000, "H": 13000}.items():
    axes[2].axhline(threshold, color="#495057", linestyle=":", linewidth=1)
    axes[2].text(5, threshold, f"seuil {quality}", fontsize=7, va="bottom")
axes[2].set_xlabel("usure outil (min)")
axes[2].set_ylabel("usure x couple (min.N.m)")
axes[2].set_title("Usure (> 200 min) et surcharge (seuil par gamme)")
handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=colour, label=label)
    for label, colour in palette.items()
]
fig.legend(
    handles=handles, loc="lower center", ncol=len(handles), fontsize=8, bbox_to_anchor=(0.5, -0.04)
)
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Chaque mode occupe une **zone** délimitée par un seuil physique sur une combinaison de capteurs — jamais sur un capteur seul. C'est ce qui justifie les trois features déclarées dans `conf/preprocessing/default.yaml` (écart de température, puissance, usure x couple).
- Les zones se chevauchent : un outil usé sous fort couple est à la fois dans la zone « usure » et dans la zone « surcharge ». Ces chevauchements annoncent les confusions de la matrice du notebook 06.
- Les défaillances aléatoires sont dispersées partout : aucune zone ne les annonce. C'est la signature d'un plafond structurel, qu'aucun modèle ne franchira.

In [ ]:
# Profil médian des capteurs par mode : la lecture tabulaire des mêmes zones.
sensors = [column for column in numeric_columns if column in raw.columns]
profile = raw.groupby(target)[sensors].median().reindex(order).round(1)
profile["alarmes"] = counts
profile

**Ce qu'il faut retenir**

- Une médiane très différente d'un mode à l'autre (usure pour `tool_wear`, couple pour `overstrain`) confirme un signal fort ; des médianes proches (`false_alarm` et `random_failure`) annoncent une confusion inévitable.
- La vibration monte avec l'usure et l'effort : c'est un signal faible mais réel, et le seul capteur absent d'une partie du parc (imputation au notebook 03).
- Ce tableau est aussi un contrôle de cohérence métier : un mode dont le profil contredirait la physique signalerait un problème d'étiquetage, pas un problème de modèle.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- Six classes déséquilibrées (de ~30 % à ~6 %) : l'accuracy récompense un modèle qui ignore les modes rares, d'où le **macro-F1** comme métrique de pilotage.
- Aucun capteur seul ne sépare les modes : ce sont des **combinaisons physiques** qui le font — l'écart process-air à basse vitesse (dissipation thermique), le produit couple × vitesse (puissance), le produit usure × couple (surcharge).
- Le seuil de surcharge dépend de la gamme (11 000 / 12 000 / 13 000 min.N.m pour L / M / H) : une règle unique, comme celle de l'automate, se trompe structurellement sur les pièces haute précision.
- `tool_wear` et `overstrain` se confondent sur les outils usés soumis à un fort couple : la paire la plus difficile du jeu, qui se lit dans la matrice de confusion.
- `heat_dissipation` et `power_failure` partagent la vitesse basse : un sous-régime de puissance ressemble à un défaut thermique tant qu'on ne regarde pas l'écart de température.
- Le code d'alarme de l'automate est un résumé bruité des capteurs : utile, mais une fois sur deux environ il désigne la mauvaise spécialité.
- `random_failure` n'a aucun signal observable par construction : son rappel reste proche de zéro pour tout modèle, oracle compris. C'est un plafond structurel, pas un défaut de réglage.
- La vibration manque sur ~4 % des alarmes (machines non équipées) : l'imputation doit être apprise sur le train et le manque lui-même n'est pas informatif.
- Les équipes de nuit acquittent un peu plus de fausses alarmes : effet faible, réel, à ne pas sur-interpréter.
- Un score parfait signerait une fuite : le générateur publie un plafond oracle calculé avec le bruit, qu'aucun modèle ne peut dépasser.

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).